# PPE Detection - YOLO26 baseline (Kaggle)
Pipeline: train -> tune confidence on VALID -> evaluate/predict on TEST -> zip & download
Kaggle: bat Internet (de pip install) va chon GPU T4 x2 (hoac P100).

## 0. Cai dat

In [ ]:
!pip install -q -U ultralytics

## 1. Cau hinh

In [ ]:
import os, glob, json, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

os.environ["WANDB_DISABLED"] = "true"

from ultralytics import YOLO, settings
settings.update({"wandb": False})

# ---- Tham so chinh (sua o day) ----
MODEL_SIZE   = "n"        # n / s / m / l / x  -> yolo26{n,s,m,l,x}.pt
EPOCHS       = 50
IMGSZ        = 640
BATCH_PER_GPU = 16        # batch tong = BATCH_PER_GPU * so GPU
PATIENCE     = 15
SEED         = 42
IOU_NMS      = 0.6        # IoU dung khi val/predict
CONF_GRID    = np.linspace(0.05, 0.95, 91)  # khong gian tim conf
EXPORT_ONNX  = True
SAVE_TEST_PREDICTIONS = True

# Co the dat tay neu tu dong tim that bai, vd "/kaggle/input/ppe-dataset/data.yaml"
DATA_YAML_SRC = None

# ---- Thiet bi / luong ----
N_GPU = torch.cuda.device_count()
DEVICE = list(range(N_GPU)) if N_GPU > 0 else "cpu"   # N_GPU>=2 -> DDP da GPU
WORKERS = os.cpu_count() or 2                           # dataloader da luong CPU
BATCH = BATCH_PER_GPU * max(N_GPU, 1)
print(f"GPUs: {N_GPU} | CPU workers: {WORKERS} | device={DEVICE} | batch={BATCH}")

OUT = Path("/kaggle/working/ppe_yolo26") if Path("/kaggle").exists() else Path("./ppe_yolo26")
OUT.mkdir(parents=True, exist_ok=True)

## 2. Tim data.yaml va tao ban sao voi duong dan tuyet doi (input cua Kaggle la read-only)

In [ ]:
if DATA_YAML_SRC is None:
    cands = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
    assert cands, "Khong tim thay data.yaml trong /kaggle/input. Hay dat DATA_YAML_SRC."
    DATA_YAML_SRC = cands[0]
print("data.yaml goc:", DATA_YAML_SRC)

root = Path(DATA_YAML_SRC).parent
with open(DATA_YAML_SRC) as f:
    cfg = yaml.safe_load(f)

new_cfg = {
    "path": str(root),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": cfg["nc"],
    "names": cfg["names"],
}
DATA_YAML = str(OUT / "data_abs.yaml")
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(new_cfg, f, sort_keys=False)

for s in ["train", "valid", "test"]:
    n = len(list((root / s / "images").glob("*.*")))
    print(f"{s:6s}: {n} anh")
CLASS_NAMES = {i: n for i, n in enumerate(cfg["names"])}

## 3. Train baseline

In [ ]:
t0 = time.time()
model = YOLO(f"yolo26{MODEL_SIZE}.pt")
model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    amp=True,
    cache=False,           # doi thanh "ram" neu du RAM va dataset nho
    project=str(OUT),
    name="train",
    exist_ok=True,
    plots=True,
    verbose=True,
)
print(f"Train xong sau {(time.time()-t0)/60:.1f} phut")

BEST = OUT / "train" / "weights" / "best.pt"
assert BEST.exists(), "Khong thay best.pt"

# Sau train: dung 1 GPU cho val/predict (val khong ho tro multi-GPU)
EVAL_DEVICE = 0 if N_GPU > 0 else "cpu"
model = YOLO(str(BEST))

## 4. Params / GFLOPs

In [ ]:
info = model.info(imgsz=IMGSZ, verbose=False)   # (layers, params, grads, gflops)
n_layers, n_params, _, gflops = info
print(f"Layers={n_layers} | Params={n_params/1e6:.2f}M | GFLOPs={gflops:.2f} @ {IMGSZ}")

## 5. Ham danh gia (chi so phat hien)

In [ ]:
def run_val(split, conf, plots=False, name="val"):
    return model.val(
        data=DATA_YAML, split=split, imgsz=IMGSZ, batch=BATCH_PER_GPU * 2,
        conf=conf, iou=IOU_NMS, device=EVAL_DEVICE, workers=WORKERS,
        plots=plots, project=str(OUT), name=name, exist_ok=True, verbose=False,
    )

def at_conf(m, conf):
    """P/R/F1 theo tung lop tai dung nguong conf (noi suy tu duong cong cua lan val conf thap)."""
    px = np.asarray(m.box.px)
    k = int(np.argmin(np.abs(px - conf)))
    rows = []
    for i, c in enumerate(m.box.ap_class_index):
        p, r = float(m.box.p_curve[i, k]), float(m.box.r_curve[i, k])
        f1 = 2 * p * r / (p + r + 1e-16)
        rows.append({
            "class": CLASS_NAMES[int(c)], "Precision": p, "Recall": r, "F1": f1,
            "AP50": float(m.box.ap50[i]), "AP50-95": float(m.box.ap[i]),
        })
    df = pd.DataFrame(rows)
    overall = {
        "conf": float(conf),
        "Precision(macro)": df["Precision"].mean(), "Recall(macro)": df["Recall"].mean(),
        "F1(macro)": df["F1"].mean(),
        "mAP50": float(m.box.map50), "mAP50-95": float(m.box.map),
    }
    return df, overall

## 6. Tuning confidence tren VALID (toi uu F1 macro)

In [ ]:
# Val voi conf rat thap de co day du duong cong P-R, sau do quet nguong.
mv = run_val("val", conf=0.001, plots=True, name="valid_eval")
px = np.asarray(mv.box.px)
f1_mean = np.asarray(mv.box.f1_curve).mean(axis=0)            # F1 macro theo conf
f1_smooth = np.convolve(f1_mean, np.ones(9) / 9, mode="same")  # lam muot

tune_rows = []
for c in CONF_GRID:
    k = int(np.argmin(np.abs(px - c)))
    P = float(np.asarray(mv.box.p_curve)[:, k].mean())
    R = float(np.asarray(mv.box.r_curve)[:, k].mean())
    tune_rows.append({"conf": round(float(c), 3), "Precision": P, "Recall": R,
                      "F1": float(f1_smooth[k])})
tune_df = pd.DataFrame(tune_rows)
best_row = tune_df.loc[tune_df["F1"].idxmax()]
BEST_CONF = float(best_row["conf"])
print(f"\n>>> Conf toi uu tren VALID (max F1 macro) = {BEST_CONF:.3f}")
print(best_row.to_string())

(OUT / "eval").mkdir(exist_ok=True)
tune_df.to_csv(OUT / "eval" / "valid_conf_sweep.csv", index=False)

# Ve duong F1-conf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tune_df["conf"], tune_df["Precision"], label="Precision")
ax.plot(tune_df["conf"], tune_df["Recall"], label="Recall")
ax.plot(tune_df["conf"], tune_df["F1"], label="F1", lw=2)
ax.axvline(BEST_CONF, ls="--", c="k", label=f"best conf={BEST_CONF:.2f}")
ax.set_xlabel("confidence"); ax.set_ylabel("score (macro)"); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(OUT / "eval" / "valid_conf_tuning.png", dpi=150); plt.close(fig)

valid_df, valid_overall = at_conf(mv, BEST_CONF)

## 7. Danh gia tren TEST voi conf da chon

In [ ]:
mt = run_val("test", conf=0.001, plots=True, name="test_eval")
test_df, test_overall = at_conf(mt, BEST_CONF)
speed = {k: round(float(v), 2) for k, v in mt.speed.items()}   # ms/anh

print("\n=== VALID @ conf=%.3f ===" % BEST_CONF)
print(pd.Series(valid_overall).round(4).to_string())
print("\n=== TEST @ conf=%.3f ===" % BEST_CONF)
print(pd.Series(test_overall).round(4).to_string())
print("\nPer-class (TEST):")
print(test_df.round(4).to_string(index=False))
print("\nTrung binh toc do (ms/anh):", speed)

valid_df.to_csv(OUT / "eval" / "valid_per_class.csv", index=False)
test_df.to_csv(OUT / "eval" / "test_per_class.csv", index=False)

summary = {
    "model": f"yolo26{MODEL_SIZE}", "imgsz": IMGSZ, "epochs_requested": EPOCHS,
    "params_M": round(n_params / 1e6, 3), "GFLOPs": round(float(gflops), 2),
    "best_conf_from_valid": BEST_CONF, "iou_nms": IOU_NMS,
    "valid": valid_overall, "test": test_overall,
    "speed_ms_per_image_test": speed,
}
with open(OUT / "eval" / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

## 8. Predict tren TEST (luu anh co ve box + file nhan txt)

In [ ]:
if SAVE_TEST_PREDICTIONS:
    test_dir = root / "test" / "images"
    results = model.predict(
        source=str(test_dir), conf=BEST_CONF, iou=IOU_NMS, imgsz=IMGSZ,
        device=EVAL_DEVICE, batch=BATCH_PER_GPU * 2, stream=True,   # stream: tiet kiem RAM
        save=True, save_txt=True, save_conf=True,
        project=str(OUT), name="test_predictions", exist_ok=True, verbose=False,
    )
    n = sum(1 for _ in results)
    print(f"Da predict {n} anh test -> {OUT/'test_predictions'}")

## 9. (Tuy chon) Export ONNX

In [ ]:
if EXPORT_ONNX:
    try:
        onnx_path = model.export(format="onnx", imgsz=IMGSZ, simplify=True)
        shutil.copy(onnx_path, OUT / "best.onnx")
    except Exception as e:
        print("Export ONNX that bai (bo qua):", e)

shutil.copy(BEST, OUT / "best.pt")
shutil.copy(DATA_YAML_SRC, OUT / "data_original.yaml")

## 10. Dong goi .zip va tao link tai ve

In [ ]:
ZIP_BASE = str(OUT.parent / "ppe_yolo26_baseline")
zip_path = shutil.make_archive(ZIP_BASE, "zip", root_dir=OUT.parent, base_dir=OUT.name)
print(f"ZIP: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")

try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(zip_path, "/kaggle/working")
                     if zip_path.startswith("/kaggle/working") else zip_path))
except Exception:
    pass
# Tren Kaggle: sau khi chay xong, tab "Output" (ben phai) -> tai ppe_yolo26_baseline.zip
# (hoac "Save Version" -> Save & Run All de file nam trong Output cua notebook).